In [1]:
import pandas as pd
import numpy as np
import itertools
import time
from dateutil.relativedelta import relativedelta
from scipy.optimize import minimize
from scipy.stats import norm

import joblib
import os
from pathlib import Path

In [2]:
print (os.getcwd())

D:\Aundrila_seminar\seminar_team_ranking\notebooks


In [4]:
df = pd.read_csv("../dataset/results.csv", parse_dates=["date"])

In [5]:
print(f"Total matches in dataset: {len(df)}")
print(f"Date range: {df['date'].min().date()} to {df['date'].max().date()}")
print(f"Unique tournaments: {df['tournament'].nunique()}")
df.head()

Total matches in dataset: 49071
Date range: 1872-11-30 to 2026-01-26
Unique tournaments: 191


,date,home_team,away_team,home_score,away_score,tournament,city,country,neutral
0,1872-11-30,Scotland,England,0,0,Friendly,Glasgow,Scotland,False
1,1873-03-08,England,Scotland,4,2,Friendly,London,England,False
2,1874-03-07,Scotland,England,2,1,Friendly,Glasgow,Scotland,False
3,1875-03-06,England,Scotland,2,2,Friendly,London,England,False
4,1876-03-04,Scotland,England,3,0,Friendly,Glasgow,Scotland,False


### 3. Tournament Mapping and Filtering
#
4 categories (clean international only):
 - `friendly` (0) — international friendlies
 - `qualifier` (1) — official qualifying campaigns
 - `confederation` (2) — 6 major continental championships only
 - `worldcup` (3) — FIFA World Cup finals

In [6]:
tournament_mapping = {
    # Friendly
    "Friendly": "friendly",

    # World Cup finals
    "FIFA World Cup": "worldcup",

    # Qualifiers
    "FIFA World Cup qualification": "qualifier",
    "AFC Asian Cup qualification": "qualifier",
    "African Cup of Nations qualification": "qualifier",
    "UEFA Euro qualification": "qualifier",
    "Copa América qualification": "qualifier",
    "Gold Cup qualification": "qualifier",
    "Oceania Nations Cup qualification": "qualifier",

    # Confederation — 6 major continental championships only
    "AFC Asian Cup": "confederation",
    "African Cup of Nations": "confederation",
    "UEFA Euro": "confederation",
    "Copa América": "confederation",
    "Gold Cup": "confederation",
    "Oceania Nations Cup": "confederation",
}

category_to_code = {
    "friendly": 0,
    "qualifier": 1,
    "confederation": 2,
    "worldcup": 3,
}

In [7]:
# Filter to last 8 years
last_date = df["date"].max()
start_date = last_date - relativedelta(years=8)
df_8y = df[df["date"] >= start_date].copy().reset_index(drop=True)
print(f"8-year window: {start_date.date()} to {last_date.date()}")
print(f"Total matches in 8-year window: {len(df_8y)}")

8-year window: 2018-01-26 to 2026-01-26
Total matches in 8-year window: 7800


In [8]:
# Apply mapping
df_8y["match_type"] = df_8y["tournament"].map(tournament_mapping)

dropped = df_8y[df_8y["match_type"].isna()]
kept = df_8y[df_8y["match_type"].notna()]

print(f"\nKept: {len(kept)} matches")
print(f"Dropped: {len(dropped)} matches\n")


Kept: 5789 matches
Dropped: 2011 matches



In [9]:
print("Dropped tournaments:")
for t, count in dropped["tournament"].value_counts().sort_values(ascending=False).items():
    print(f"    - {t} ({count} matches)")

# %%
df_8y = kept.copy().reset_index(drop=True)
df_8y["category"] = df_8y["match_type"].map(category_to_code)

Dropped tournaments:
    - UEFA Nations League (658 matches)
    - CONCACAF Nations League (422 matches)
    - COSAFA Cup (112 matches)
    - AFF Championship (78 matches)
    - CONCACAF Nations League qualification (68 matches)
    - Arab Cup (63 matches)
    - Pacific Games (46 matches)
    - CONIFA World Football Cup (45 matches)
    - Gulf Cup (45 matches)
    - Island Games (44 matches)
    - SAFF Cup (38 matches)
    - ASEAN Championship (26 matches)
    - FIFA Series (24 matches)
    - EAFF Championship (24 matches)
    - Indian Ocean Island Games (23 matches)
    - MSG Prime Minister's Cup (23 matches)
    - CONIFA European Football Cup (20 matches)
    - CONCACAF Series (19 matches)
    - Inter Games (19 matches)
    - Intercontinental Cup (17 matches)
    - WAFF Championship (17 matches)
    - King's Cup (16 matches)
    - Kirin Challenge Cup (14 matches)
    - Baltic Cup (14 matches)
    - Arab Cup qualification (13 matches)
    - CAFA Nations Cup (10 matches)
    - Kirin Cu

In [10]:
print("\nCategory breakdown:")
for match_type, code in category_to_code.items():
    matches = df_8y[df_8y["category"] == code]
    tournaments = sorted(matches["tournament"].unique())
    print(f"\n  [{code}] {match_type}: {len(matches)} matches")
    for t in tournaments:
        c = (matches["tournament"] == t).sum()
        print(f"       - {t} ({c})")


Category breakdown:

  [0] friendly: 2059 matches
       - Friendly (2059)

  [1] qualifier: 2967 matches
       - AFC Asian Cup qualification (118)
       - African Cup of Nations qualification (560)
       - Copa América qualification (2)
       - FIFA World Cup qualification (1751)
       - Gold Cup qualification (32)
       - Oceania Nations Cup qualification (3)
       - UEFA Euro qualification (501)

  [2] confederation: 635 matches
       - AFC Asian Cup (102)
       - African Cup of Nations (208)
       - Copa América (86)
       - Gold Cup (124)
       - Oceania Nations Cup (13)
       - UEFA Euro (102)

  [3] worldcup: 128 matches
       - FIFA World Cup (128)


In [11]:
print(f"\nTotal matches for modeling: {len(df_8y)}")


Total matches for modeling: 5789


### 4. Precompute All Constants
#
 Thurstone-Mosteller is outcome-based:
 - No prediction grid needed (no scorelines)
 - Only needs outcome codes: home_win=1, draw=0, away_win=-1
 - Probabilities computed directly from normal CDF
 - Draw region: |strength_diff| < δ (draw threshold)

In [12]:
# --- Category counts (for reference) ---
print("Category counts:")
print(f"  {'Category':<16} {'Count':<10} {'% of total'}")
print(f"  {'─' * 40}")
total_matches_all = len(df_8y)
for name, code in category_to_code.items():
    count = int((df_8y["category"] == code).sum())
    pct = count / total_matches_all * 100
    print(f"  {name:<16} {count:<10} {pct:.1f}%")
print(f"  {'─' * 40}")
print(f"  {'Total':<16} {total_matches_all:<10}")

Category counts:
  Category         Count      % of total
  ────────────────────────────────────────
  friendly         2059       35.6%
  qualifier        2967       51.3%
  confederation    635        11.0%
  worldcup         128        2.2%
  ────────────────────────────────────────
  Total            5789      


In [13]:
# ## 5. Precompute Window Caches

# %%
def precompute_window(train_df, test_df, halfperiod=3.5 * 365.25):
    """
    Precompute all constants for a single rolling window.
    Called once per window, reused across all weight combinations.

    Thurstone-Mosteller: outcome-based, no scorelines needed.
    halfperiod=3.5 years (paper's optimal for TM).
    """
    # --- Training cache ---
    teams = sorted(set(train_df["home_team"]).union(set(train_df["away_team"])))
    team_to_idx = {t: i for i, t in enumerate(teams)}
    K = len(teams)
    n = len(train_df)

    h_indices = np.array([team_to_idx[t] for t in train_df["home_team"].values])
    a_indices = np.array([team_to_idx[t] for t in train_df["away_team"].values])
    home_scores = train_df["home_score"].values.astype(int)
    away_scores = train_df["away_score"].values.astype(int)
    not_neutral = ~train_df["neutral"].values.astype(bool)
    categories = train_df["category"].values.astype(int)

    # Outcome codes: 1=home win, 0=draw, -1=away win
    outcomes = np.sign(home_scores - away_scores).astype(int)

    # Precompute boolean masks for each outcome (avoids branching in likelihood)
    is_home_win = outcomes == 1
    is_draw = outcomes == 0
    is_away_win = outcomes == -1

    # Temporal decay weights
    end_date = train_df["date"].max()
    days_ago = (end_date - train_df["date"]).dt.days.values
    w_time = 0.5 ** (days_ago / halfperiod)

    train_cache = {
        "teams": teams,
        "team_to_idx": team_to_idx,
        "K": K,
        "n": n,
        "h_indices": h_indices,
        "a_indices": a_indices,
        "not_neutral": not_neutral,
        "categories": categories,
        "is_home_win": is_home_win,
        "is_draw": is_draw,
        "is_away_win": is_away_win,
        "w_time": w_time,
    }

    # --- Test cache ---
    t_h_idx = []
    t_a_idx = []
    t_hs = []
    t_as = []
    t_nn = []

    for _, row in test_df.iterrows():
        ht = row["home_team"]
        at = row["away_team"]
        if ht in team_to_idx and at in team_to_idx:
            t_h_idx.append(team_to_idx[ht])
            t_a_idx.append(team_to_idx[at])
            t_hs.append(int(row["home_score"]))
            t_as.append(int(row["away_score"]))
            t_nn.append(not bool(row["neutral"]))

    n_valid = len(t_h_idx)

    # Precompute actual cumulative outcome vectors for RPS
    t_hs_arr = np.array(t_hs, dtype=int)
    t_as_arr = np.array(t_as, dtype=int)
    cum_actual = np.zeros((n_valid, 2))
    for i in range(n_valid):
        if t_hs_arr[i] > t_as_arr[i]:
            cum_actual[i] = [1.0, 1.0]
        elif t_hs_arr[i] == t_as_arr[i]:
            cum_actual[i] = [0.0, 1.0]
        else:
            cum_actual[i] = [0.0, 0.0]

    test_cache = {
        "h_indices": np.array(t_h_idx, dtype=int),
        "a_indices": np.array(t_a_idx, dtype=int),
        "not_neutral": np.array(t_nn, dtype=bool),
        "n_valid": n_valid,
        "cum_actual": cum_actual,
    }

    return train_cache, test_cache

In [14]:
def get_rolling_windows(df):
    first_date = df["date"].min()
    last_date = df["date"].max()
    train_start = first_date.replace(day=1)
    windows = []
    while True:
        test_start = train_start + relativedelta(months=11)
        test_end = test_start + relativedelta(months=1)
        if test_end > last_date + pd.Timedelta(days=1):
            break
        train_mask = (df["date"] >= train_start) & (df["date"] < test_start)
        test_mask = (df["date"] >= test_start) & (df["date"] < test_end)
        train_df = df[train_mask]
        test_df = df[test_mask]
        if len(train_df) > 0 and len(test_df) > 0:
            info = {
                "train_start": train_start,
                "train_end": test_start,
                "test_start": test_start,
                "test_end": test_end,
                "train_matches": len(train_df),
                "test_matches": len(test_df),
            }
            windows.append((train_df.copy(), test_df.copy(), info))
        train_start += relativedelta(months=1)
    return windows

In [15]:
print("Precomputing window caches...")
precompute_start = time.time()

Precomputing window caches...


In [16]:
raw_windows = get_rolling_windows(df_8y)
window_caches = []
for train_df, test_df, info in raw_windows:
    train_cache, test_cache = precompute_window(train_df, test_df)
    window_caches.append((train_cache, test_cache, info))

In [17]:
precompute_time = time.time() - precompute_start
print(f"Precomputed {len(window_caches)} windows in {precompute_time:.1f}s\n")

Precomputed 73 windows in 15.4s



In [18]:
print("First 5 windows:")
for i, (_, _, info) in enumerate(window_caches[:5]):
    print(f"  Window {i+1}: Train {info['train_start'].strftime('%Y-%m')} to "
          f"{info['train_end'].strftime('%Y-%m')} ({info['train_matches']} matches) → "
          f"Test {info['test_start'].strftime('%Y-%m')} ({info['test_matches']} matches)")

First 5 windows:
  Window 1: Train 2018-01 to 2018-12 (571 matches) → Test 2018-12 (24 matches)
  Window 2: Train 2018-02 to 2019-01 (589 matches) → Test 2019-01 (56 matches)
  Window 3: Train 2018-03 to 2019-02 (642 matches) → Test 2019-02 (5 matches)
  Window 4: Train 2018-04 to 2019-03 (510 matches) → Test 2019-03 (121 matches)
  Window 5: Train 2018-06 to 2019-05 (586 matches) → Test 2019-05 (7 matches)


In [19]:
print(f"\nLast window:")
_, _, info = window_caches[-1]
print(f"  Window {len(window_caches)}: Train {info['train_start'].strftime('%Y-%m')} to "
      f"{info['train_end'].strftime('%Y-%m')} ({info['train_matches']} matches) → "
      f"Test {info['test_start'].strftime('%Y-%m')} ({info['test_matches']} matches)")


Last window:
  Window 73: Train 2025-01 to 2025-12 (816 matches) → Test 2025-12 (36 matches)


### 6. Thurstone-Mosteller Model
#
**How it works:**
 - Each team i has strength sᵢ
 - Performance in a match: Pᵢ ~ N(sᵢ, 1)
 - Home team gets advantage: h_effect added to strength
 - Strength difference: d = (s_home + h_effect) - s_away
 - Draw threshold δ: match is a draw if |performance diff| < δ
#
**Probabilities (using normal CDF Φ):**
 - P(home win) = 1 - Φ(δ - d)  = Φ(d - δ)
 - P(draw)     = Φ(δ - d) - Φ(-δ - d)
 - P(away win) = Φ(-δ - d)

 **Parameters:**
 - params[0] = home advantage
 - params[1] = log(δ) — draw threshold (log to keep positive)
 - params[2:2+K] = team abilities

 No constant term needed — only differences matter.

In [20]:
# Precompute normal CDF/logCDF for speed
NORM_CDF = norm.cdf
NORM_LOGCDF = norm.logcdf

# Small epsilon for numerical stability
EPS = 1e-15

In [21]:
def tm_log_probs(d, delta):
    """
    Compute log-probabilities for all three outcomes.

    P(home win) = Φ(d - δ)
    P(draw)     = Φ(δ - d) - Φ(-δ - d) = Φ(δ - d) - Φ(-(δ + d))
    P(away win) = Φ(-δ - d) = Φ(-(δ + d))

    Args:
        d: array of strength differences (s_home + h - s_away)
        delta: draw threshold (scalar > 0)

    Returns:
        log_p_hw, log_p_draw, log_p_aw: arrays of log-probabilities
    """
    # Precompute CDF values (each computed once, reused)
    cdf_d_minus_delta = NORM_CDF(d - delta)        # Φ(d - δ)
    cdf_neg_delta_minus_d = NORM_CDF(-delta - d)    # Φ(-(δ + d))

    p_hw = np.maximum(cdf_d_minus_delta, EPS)
    p_aw = np.maximum(cdf_neg_delta_minus_d, EPS)
    p_draw = np.maximum(1.0 - p_hw - p_aw, EPS)

    return np.log(p_hw), np.log(p_draw), np.log(p_aw)

In [22]:
def fit_from_cache(train_cache, mi_weights, warm_start=None):
    """
    Fit Thurstone-Mosteller using precomputed cache.

    Parameters:
        params[0] = home advantage
        params[1] = log(δ) — draw threshold
        params[2:2+K] = team abilities

    """
    K = train_cache["K"]
    h_indices = train_cache["h_indices"]
    a_indices = train_cache["a_indices"]
    not_neutral = train_cache["not_neutral"]
    is_home_win = train_cache["is_home_win"]
    is_draw = train_cache["is_draw"]
    is_away_win = train_cache["is_away_win"]
    w_time = train_cache["w_time"]
    categories = train_cache["categories"]

    # temporal × importance only
    w_importance = np.array([mi_weights[c] for c in categories])
    w = w_time * w_importance

    def neg_log_lik(params):
        home_adv = params[0]
        log_delta = params[1]
        abilities = params[2:2 + K]

        delta = np.exp(np.clip(log_delta, -5, 3))

        h_effect = np.where(not_neutral, home_adv, 0.0)
        d = abilities[h_indices] - abilities[a_indices] + h_effect

        log_p_hw, log_p_draw, log_p_aw = tm_log_probs(d, delta)

        # Select log-probability for actual outcome
        log_probs = (
            is_home_win * log_p_hw
            + is_draw * log_p_draw
            + is_away_win * log_p_aw
        )

        return -np.sum(w * log_probs)

    # 2 global params + K abilities
    if warm_start is not None and len(warm_start) == 2 + K:
        params0 = warm_start.copy()
    else:
        params0 = np.zeros(2 + K)
        params0[0] = 0.1           # small home advantage
        params0[1] = np.log(0.3)   # initial draw threshold

    result = minimize(
        neg_log_lik,
        params0,
        method="L-BFGS-B",
        options={"maxiter": 3000, "ftol": 1e-9},
    )

    params = result.x
    abilities = params[2:2 + K]
    abilities = abilities - abilities.mean()

    return {
        "home": params[0],
        "delta": np.exp(np.clip(params[1], -5, 3)),
        "abilities_arr": abilities,
        "raw_params": params,
    }

In [23]:
def evaluate_from_cache(model, test_cache):
    """
    Evaluate model on test set — returns (sum_rps, count).

    Thurstone-Mosteller predicts P(home win), P(draw), P(away win)
    directly — no prediction grid needed.
    """
    n_valid = test_cache["n_valid"]
    if n_valid == 0:
        return 0.0, 0

    h_indices = test_cache["h_indices"]
    a_indices = test_cache["a_indices"]
    not_neutral = test_cache["not_neutral"]
    cum_actual = test_cache["cum_actual"]

    home_adv = model["home"]
    delta = model["delta"]
    abilities = model["abilities_arr"]

    h_effect = np.where(not_neutral, home_adv, 0.0)
    d = abilities[h_indices] - abilities[a_indices] + h_effect

    # Compute probabilities for all test matches at once (vectorized)
    cdf_d_minus_delta = NORM_CDF(d - delta)
    cdf_neg_delta_minus_d = NORM_CDF(-delta - d)

    p_hw = np.maximum(cdf_d_minus_delta, EPS)
    p_aw = np.maximum(cdf_neg_delta_minus_d, EPS)
    p_draw = np.maximum(1.0 - p_hw - p_aw, EPS)

    # Normalize
    total = p_hw + p_draw + p_aw
    p_hw = p_hw / total
    p_draw = p_draw / total
    p_aw = p_aw / total

    # RPS — fully vectorized (no per-match loop!)
    cum_pred_0 = p_hw
    cum_pred_1 = p_hw + p_draw

    sum_rps = 0.5 * np.sum(
        (cum_pred_0 - cum_actual[:, 0]) ** 2
        + (cum_pred_1 - cum_actual[:, 1]) ** 2
    )

    return sum_rps, n_valid

### 7. Evaluate Paper's Baseline


In [24]:
def evaluate_weights(window_caches, mi_weights, warm_starts=None, verbose=False):
    """Run a weight combination across all windows with warm starting."""
    total_rps = 0.0
    total_matches = 0
    new_warm_starts = {}
    failed_windows = 0

    for w_idx, (train_cache, test_cache, info) in enumerate(window_caches):
        try:
            ws = warm_starts.get(w_idx) if warm_starts else None
            model = fit_from_cache(train_cache, mi_weights, warm_start=ws)
            new_warm_starts[w_idx] = model["raw_params"]

            s_rps, n = evaluate_from_cache(model, test_cache)
            total_rps += s_rps
            total_matches += n

            if verbose and (w_idx + 1) % 10 == 0:
                running_avg = total_rps / total_matches if total_matches > 0 else 0
                print(f"      Window {w_idx+1}/{len(window_caches)} | "
                      f"running avg RPS: {running_avg:.6f} | "
                      f"matches so far: {total_matches}")
        except Exception as e:
            failed_windows += 1
            if verbose:
                print(f"      Window {w_idx+1} FAILED: {e}")
            continue

    if verbose and failed_windows > 0:
        print(f"      ⚠ {failed_windows} windows failed")

    avg_rps = total_rps / total_matches if total_matches > 0 else np.nan
    return avg_rps, total_matches, new_warm_starts

In [25]:
print("-" * 60)
print("THURSTONE-MOSTELLER MODEL")
print("-" * 60)
print("Evaluating paper's baseline weights...")
paper_weights = [1.0, 2.5, 3.0, 4.0]
paper_rps, paper_n, _ = evaluate_weights(window_caches, paper_weights, verbose=True)
print(f"\nPaper's weights:     [friendly=1.0, qualifier=2.5, confederation=3.0, worldcup=4.0]")
print(f"Paper's Avg RPS:     {paper_rps:.6f}")
print(f"Matches evaluated:   {paper_n}")

------------------------------------------------------------
THURSTONE-MOSTELLER MODEL
------------------------------------------------------------
Evaluating paper's baseline weights...
      Window 10/73 | running avg RPS: 0.265530 | matches so far: 679
      Window 20/73 | running avg RPS: 0.278363 | matches so far: 1116
      Window 30/73 | running avg RPS: 0.257186 | matches so far: 1973
      Window 40/73 | running avg RPS: 0.259356 | matches so far: 2468
      Window 50/73 | running avg RPS: 0.259231 | matches so far: 3147
      Window 60/73 | running avg RPS: 0.255060 | matches so far: 3783
      Window 70/73 | running avg RPS: 0.256650 | matches so far: 4427

Paper's weights:     [friendly=1.0, qualifier=2.5, confederation=3.0, worldcup=4.0]
Paper's Avg RPS:     0.252194
Matches evaluated:   4768


### 8. Grid Search with Warm Starting

In [26]:
friendly_vals      = [0.75, 1.00, 1.25, 1.50, 1.75]
qualifier_vals     = [1.50, 1.75, 2.00, 2.25, 2.50, 2.75]
confederation_vals = [1.50, 1.75, 2.00, 2.25, 2.50, 2.75, 3.00]
worldcup_vals      = [3.5, 4.0, 4.25, 4.5, 5.0]

In [27]:
grid = list(itertools.product(
    friendly_vals,
    qualifier_vals,
    confederation_vals,
    worldcup_vals,
))

In [28]:
total_combos = len(grid)
total_windows = len(window_caches)
total_fits = total_combos * total_windows

In [29]:
print(f"\nGrid search configuration:")
print(f"  Friendly values:      {friendly_vals}")
print(f"  Qualifier values:     {qualifier_vals}")
print(f"  Confederation values: {confederation_vals}")
print(f"  World Cup values:     {worldcup_vals}")
print(f"  ─────────────────────────────────")
print(f"  Weight combinations:  {total_combos}")
print(f"  Rolling windows:      {total_windows}")
print(f"  Total model fits:     {total_fits}")


Grid search configuration:
  Friendly values:      [0.75, 1.0, 1.25, 1.5, 1.75]
  Qualifier values:     [1.5, 1.75, 2.0, 2.25, 2.5, 2.75]
  Confederation values: [1.5, 1.75, 2.0, 2.25, 2.5, 2.75, 3.0]
  World Cup values:     [3.5, 4.0, 4.25, 4.5, 5.0]
  ─────────────────────────────────
  Weight combinations:  1050
  Rolling windows:      73
  Total model fits:     76650


In [30]:
results = []
start_time = time.time()
prev_warm_starts = None

best_rps_so_far = np.inf
best_weights_so_far = None
completed = 0
failed_count = 0

In [50]:
print("GRID SEARCH STARTED — THURSTONE-MOSTELLER ")

print(f"Start time: {time.strftime('%Y-%m-%d %H:%M:%S')}")
print(f"Paper's baseline: RPS = {paper_rps:.6f}")


GRID SEARCH STARTED — THURSTONE-MOSTELLER 
Start time: 2026-03-18 19:45:21
Paper's baseline: RPS = 0.252194


In [32]:
for idx, weights in enumerate(grid):
    combo_start = time.time()

    avg_rps, n_matches, prev_warm_starts = evaluate_weights(
        window_caches, list(weights), warm_starts=prev_warm_starts
    )

    combo_time = time.time() - combo_start
    completed += 1

    if np.isnan(avg_rps):
        failed_count += 1

    results.append({
        "friendly": weights[0],
        "qualifier": weights[1],
        "confederation": weights[2],
        "worldcup": weights[3],
        "avg_rps": avg_rps,
        "n_matches": n_matches,
    })

    # Track best
    new_best = False
    if not np.isnan(avg_rps) and avg_rps < best_rps_so_far:
        best_rps_so_far = avg_rps
        best_weights_so_far = weights
        new_best = True

    # Progress
    elapsed = time.time() - start_time
    avg_per_combo = elapsed / completed
    remaining = avg_per_combo * (total_combos - completed)
    pct = completed / total_combos * 100
    eta = time.strftime('%H:%M:%S', time.gmtime(remaining))

    # Log every 20 combos, on new best, or at the end
    if completed % 20 == 0 or new_best or completed == total_combos:
        status = "★ NEW BEST" if new_best else ""
        print(
            f"  [{completed:>4d}/{total_combos}] {pct:5.1f}% | "
            f"weights=[{weights[0]:.2f}, {weights[1]:.2f}, {weights[2]:.2f}, {weights[3]:.2f}] | "
            f"RPS={avg_rps:.6f} | "
            f"best={best_rps_so_far:.6f} | "
            f"this={combo_time:.1f}s | "
            f"elapsed={elapsed/60:.1f}min | "
            f"ETA={eta} {status}"
        )

total_time = time.time() - start_time

  [   1/1050]   0.1% | weights=[0.75, 1.50, 1.50, 3.50] | RPS=0.250993 | best=0.250993 | this=171.6s | elapsed=3.1min | ETA=06:20:07 ★ NEW BEST
  [  20/1050]   1.9% | weights=[0.75, 1.50, 2.25, 5.00] | RPS=0.254124 | best=0.250993 | this=45.3s | elapsed=33.0min | ETA=04:18:26 
  [  40/1050]   3.8% | weights=[0.75, 1.75, 1.50, 5.00] | RPS=0.254106 | best=0.250993 | this=36.6s | elapsed=51.8min | ETA=21:46:46 
  [  60/1050]   5.7% | weights=[0.75, 1.75, 2.50, 5.00] | RPS=0.255080 | best=0.250993 | this=28.1s | elapsed=67.6min | ETA=18:35:28 
  [  80/1050]   7.6% | weights=[0.75, 2.00, 1.75, 5.00] | RPS=0.255001 | best=0.250993 | this=27.8s | elapsed=84.0min | ETA=16:58:22 
  [ 100/1050]   9.5% | weights=[0.75, 2.00, 2.75, 5.00] | RPS=0.255898 | best=0.250993 | this=27.8s | elapsed=99.2min | ETA=15:41:58 
  [ 120/1050]  11.4% | weights=[0.75, 2.25, 2.00, 5.00] | RPS=0.255842 | best=0.250993 | this=29.3s | elapsed=115.7min | ETA=14:56:43 
  [ 140/1050]  13.3% | weights=[0.75, 2.25, 3.00, 5

In [33]:
print(f"  Best RPS:         {best_rps_so_far:.6f}")
print(f"  Best weights:     [{best_weights_so_far[0]:.2f}, {best_weights_so_far[1]:.2f}, "
      f"{best_weights_so_far[2]:.2f}, {best_weights_so_far[3]:.2f}]")

  Best RPS:         0.250993
  Best weights:     [0.75, 1.50, 1.50, 3.50]


### Results

In [51]:
results_df = pd.DataFrame(results).dropna(subset=["avg_rps"])
results_df = results_df.sort_values("avg_rps").reset_index(drop=True)


In [52]:
results_df.to_csv("grid_search_thurstone_mosteller_results.csv", index=False)
print("Full results saved to grid_search_thurstone_mosteller_results.csv\n")

Full results saved to grid_search_thurstone_mosteller_results.csv



In [37]:
print("TOP 10 WEIGHT COMBINATIONS — THURSTONE-MOSTELLER (lowest RPS = best)")
print("-" * 95)
print(f"{'Rank':<6} {'Friendly':<10} {'Qualifier':<10} {'Confederation':<14} {'WorldCup':<10} {'Avg RPS':<12} {'Matches':<8}")
print("-" * 70)
for i, row in results_df.head(10).iterrows():
    print(f"{i+1:<6} {row['friendly']:<10.2f} {row['qualifier']:<10.2f} "
          f"{row['confederation']:<14.2f} {row['worldcup']:<10.2f} "
          f"{row['avg_rps']:<12.6f} {int(row['n_matches']):<8}")

TOP 10 WEIGHT COMBINATIONS — THURSTONE-MOSTELLER (lowest RPS = best)
-----------------------------------------------------------------------------------------------
Rank   Friendly   Qualifier  Confederation  WorldCup   Avg RPS      Matches 
----------------------------------------------------------------------
1      0.75       1.50       1.50           3.50       0.250993     4768    
2      1.75       1.50       1.50           5.00       0.251448     4768    
3      1.75       1.50       1.50           4.50       0.251449     4768    
4      1.75       1.50       1.50           4.25       0.251454     4768    
5      1.75       1.50       1.50           4.00       0.251460     4768    
6      1.50       1.50       1.50           4.00       0.251463     4768    
7      1.50       1.50       1.50           4.25       0.251465     4768    
8      1.50       1.50       1.50           4.50       0.251465     4768    
9      1.50       1.50       1.50           5.00       0.251474     476

In [38]:
print("\nBOTTOM 10 (worst combinations):")
print(f"{'Rank':<6} {'Friendly':<10} {'Qualifier':<10} {'Confederation':<14} {'WorldCup':<10} {'Avg RPS':<12}")
print("-" * 72)
for i, row in results_df.tail(10).iterrows():
    print(f"{i+1:<6} {row['friendly']:<10.2f} {row['qualifier']:<10.2f} "
          f"{row['confederation']:<14.2f} {row['worldcup']:<10.2f} "
          f"{row['avg_rps']:<12.6f}")


BOTTOM 10 (worst combinations):
Rank   Friendly   Qualifier  Confederation  WorldCup   Avg RPS     
------------------------------------------------------------------------
1041   0.75       2.75       2.75           4.00       0.257493    
1042   0.75       2.75       2.75           4.25       0.257499    
1043   0.75       2.75       2.75           3.50       0.257501    
1044   0.75       2.75       2.75           4.50       0.257508    
1045   0.75       2.75       2.75           5.00       0.257527    
1046   0.75       2.75       3.00           4.00       0.257673    
1047   0.75       2.75       3.00           4.25       0.257678    
1048   0.75       2.75       3.00           3.50       0.257678    
1049   0.75       2.75       3.00           4.50       0.257685    
1050   0.75       2.75       3.00           5.00       0.257705    


In [39]:
best = results_df.iloc[0]

In [48]:
print("FINAL COMPARISON — THURSTONE-MOSTELLER")


print(f"\n  Paper's weights:")
print(f"    Friendly:       1.00")
print(f"    Qualifier:      2.50")
print(f"    Confederation:  3.00")
print(f"    World Cup:      4.00")
print(f"    Avg RPS:        {paper_rps:.6f}")

print(f"\n  Best weights found:")
print(f"    Friendly:       {best['friendly']:.2f}")
print(f"    Qualifier:      {best['qualifier']:.2f}")
print(f"    Confederation:  {best['confederation']:.2f}")
print(f"    World Cup:      {best['worldcup']:.2f}")
print(f"    Avg RPS:        {best['avg_rps']:.6f}")

FINAL COMPARISON — THURSTONE-MOSTELLER

  Paper's weights:
    Friendly:       1.00
    Qualifier:      2.50
    Confederation:  3.00
    World Cup:      4.00
    Avg RPS:        0.252194

  Best weights found:
    Friendly:       0.75
    Qualifier:      1.50
    Confederation:  1.50
    World Cup:      3.50
    Avg RPS:        0.250993


In [44]:
improvement_vs_paper = ((paper_rps - best["avg_rps"]) / paper_rps) * 100
print(f"\n new approach vs Paper's weights: {improvement_vs_paper:+.4f}%")


 new approach vs Paper's weights: +0.4764%


In [45]:
if improvement_vs_paper > 0:
    print("New Weights are BETTER than the paper's!")
elif improvement_vs_paper < 0:
    print("The paper's weights perform better.")
else:
    print("  = Same performance.")

New Weights are BETTER than the paper's!
